In [ ]:
# Cell 1: GPU check
# Ask PyTorch: can you see a GPU?
import torch
print("GPU found:", torch.cuda.is_available())

# Print the GPU's name and its memory in GB
print("Name:", torch.cuda.get_device_name(0))
print("Memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

GPU found: True
Name: Tesla T4
Memory (GB): 15.6


In [ ]:
## Cell 2: Google Drive mount

#  Plug Google Drive into Colab, at the path /content/drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
## Cell 3: Project folders on Google Drive

#  Path is a tool for folder addresses
from pathlib import Path

# The project's safe home on Drive (it stays after Colab resets)
PROJECT = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content")

# Make the four folders, one at a time
# parents=True: also make any missing folders above it, like 2026Research
# exist_ok=True: no error if the folder is already there
for name in ["data", "checkpoints", "results", "figures"]:
    (PROJECT / name).mkdir(parents=True, exist_ok=True)

# Print what is inside the project folder now
for p in sorted(PROJECT.iterdir()):
    print(p)

/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/checkpoints
/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/data
/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/figures
/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/results


In [ ]:
# Cell 4: Repository clone or update

from pathlib import Path

# Repository location on GitHub and target folder on the Colab runtime (temporary disk)
REPO_URL = "https://github.com/Prhm93/floodfm-sen1floods11.git"
REPO_DIR = Path("/content/floodfm-sen1floods11")

# Clone on the first run in a session; pull the latest commits on later runs
# git -C <folder> runs the git command as if from inside that folder
if REPO_DIR.exists():
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

Cloning into '/content/floodfm-sen1floods11'...
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 0), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), done.


In [ ]:
# Cell 5: Repository check

# Most recent commit in the cloned copy, and the files it contains
!git -C {REPO_DIR} log --oneline -1
!ls -a {REPO_DIR}

7b47c39 (HEAD -> main, origin/main, origin/HEAD) Add README and .gitignore
.  ..  .git  .gitignore  README.md


In [ ]:
# Cell 6: Official bucket inspection and Drive space check
import shutil

# Official Sen1Floods11 v1.1 bucket (source: github.com/cloudtostreet/Sen1Floods11)
BUCKET = "gs://sen1floods11/v1.1"

# Contents of the hand-labelled folder and the split folder
!gsutil ls {BUCKET}/data/flood_events/HandLabeled/
!gsutil ls {BUCKET}/splits/flood_handlabeled/

# Size of the three folders required by this project
!gsutil du -sh {BUCKET}/data/flood_events/HandLabeled/S1Hand {BUCKET}/data/flood_events/HandLabeled/S2Hand {BUCKET}/data/flood_events/HandLabeled/LabelHand

# Free space on Google Drive, in GB
total, used, free = shutil.disk_usage("/content/drive/MyDrive")
print(f"Drive free space: {free / 1e9:.1f} GB")

HandLabeled folder exists: False


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Research Idea Developement/Sen1Flood11 Dataset/HandLabeled'

In [10]:
import shutil

BUCKET = "gs://sen1floods11/v1.1"

# Contents of the hand-labelled folder and the split folder in the official bucket
!gsutil ls {BUCKET}/data/flood_events/HandLabeled/
!gsutil ls {BUCKET}/splits/flood_handlabeled/

# Size of the three folders required by this project
!gsutil du -sh {BUCKET}/data/flood_events/HandLabeled/S1Hand {BUCKET}/data/flood_events/HandLabeled/S2Hand {BUCKET}/data/flood_events/HandLabeled/LabelHand

# Free space on Google Drive, in GB
total, used, free = shutil.disk_usage("/content/drive/MyDrive")
print(f"Drive free space: {free / 1e9:.1f} GB")

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://sen1floods11/v1.1/data/flood_events/HandLabeled/JRCWaterHand/
gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/
gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/
gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1OtsuLabelHand/
gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_bolivia_data.csv
gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_test_data.csv
gs://sen1floods11/v1

In [ ]:
# Cell 7: Dataset download from the official bucket

from pathlib import Path

# Target folder for the dataset inside the project's Drive data folder
SEN1_DIR = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/data/sen1floods11")

# Download of the three hand-labelled layers; rsync copies only missing files, so a re-run resumes an interrupted download
for layer in ["S1Hand", "S2Hand", "LabelHand"]:
    target = SEN1_DIR / "HandLabeled" / layer
    target.mkdir(parents=True, exist_ok=True)
    !gsutil -m -q rsync -r {BUCKET}/data/flood_events/HandLabeled/{layer} {target}
    print("Finished:", layer)

# Download of the official split files
splits_dir = SEN1_DIR / "splits" / "flood_handlabeled"
splits_dir.mkdir(parents=True, exist_ok=True)
!gsutil -m -q rsync -r {BUCKET}/splits/flood_handlabeled {splits_dir}
print("Finished: splits")


both the source and destination. Your crcmod installation isn't using the
module's C extension, so checksumming will run very slowly. If this is your
first rsync since updating gsutil, this rsync can take significantly longer than
usual. For help installing the extension, please see "gsutil help crcmod".

Finished: S1Hand

both the source and destination. Your crcmod installation isn't using the
module's C extension, so checksumming will run very slowly. If this is your
first rsync since updating gsutil, this rsync can take significantly longer than
usual. For help installing the extension, please see "gsutil help crcmod".

Finished: S2Hand

both the source and destination. Your crcmod installation isn't using the
module's C extension, so checksumming will run very slowly. If this is your
first rsync since updating gsutil, this rsync can take significantly longer than
usual. For help installing the extension, please see "gsutil help crcmod".

Finished: LabelHand

both the source and d

In [13]:
# Cell 8: Downloaded data overview
from pathlib import Path
from collections import Counter

SEN1_DIR = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/data/sen1floods11")
HAND_DIR = SEN1_DIR / "HandLabeled"

# Number of files, file types and two example names for each layer
for layer in ["S1Hand", "S2Hand", "LabelHand"]:
    files = sorted(f for f in (HAND_DIR / layer).iterdir() if f.is_file())
    types = Counter(f.suffix for f in files)
    print(f"\n{layer}/  ->  {len(files)} files  {dict(types)}")
    for f in files[:2]:
        print("    e.g.", f.name)


S1Hand/  ->  446 files  {'.tif': 446}
    e.g. Bolivia_103757_S1Hand.tif
    e.g. Bolivia_129334_S1Hand.tif

S2Hand/  ->  446 files  {'.tif': 446}
    e.g. Bolivia_103757_S2Hand.tif
    e.g. Bolivia_129334_S2Hand.tif

LabelHand/  ->  446 files  {'.tif': 446}
    e.g. Bolivia_103757_LabelHand.tif
    e.g. Bolivia_129334_LabelHand.tif


In [14]:
# Cell 9: Split files check
from pathlib import Path

SPLITS_DIR = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/data/sen1floods11/splits/flood_handlabeled")

# Number of non-empty lines in each split file, and the first line as an example of its format
total = 0
for f in sorted(SPLITS_DIR.glob("*.csv")):
    lines = [line for line in f.read_text().splitlines() if line.strip()]
    total += len(lines)
    print(f"{f.name}: {len(lines)} lines")
    print("    first line:", lines[0])

print("\nTotal lines across all split files:", total)

flood_bolivia_data.csv: 15 lines
    first line: Bolivia_103757_S1Hand.tif,Bolivia_103757_LabelHand.tif
flood_test_data.csv: 90 lines
    first line: Ghana_313799_S1Hand.tif,Ghana_313799_LabelHand.tif
flood_train_data.csv: 252 lines
    first line: Ghana_103272_S1Hand.tif,Ghana_103272_LabelHand.tif
flood_valid_data.csv: 89 lines
    first line: Ghana_5079_S1Hand.tif,Ghana_5079_LabelHand.tif

Total lines across all split files: 446


In [15]:
# Cell 10: Split integrity check and Stage 0 data report
import json
from datetime import datetime
from pathlib import Path
from collections import Counter

SEN1_DIR = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/data/sen1floods11")
HAND_DIR = SEN1_DIR / "HandLabeled"
SPLITS_DIR = SEN1_DIR / "splits" / "flood_handlabeled"
REPORT_PATH = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/results/stage0_data_report.json")

LAYERS = ["S1Hand", "S2Hand", "LabelHand"]
SPLIT_FILES = {
    "train": "flood_train_data.csv",
    "valid": "flood_valid_data.csv",
    "test": "flood_test_data.csv",
    "bolivia": "flood_bolivia_data.csv",
}

# Chip IDs per split: the S1 file name in each line, without the "_S1Hand.tif" ending (e.g. "Ghana_103272")
splits = {}
for split_name, file_name in SPLIT_FILES.items():
    lines = [line for line in (SPLITS_DIR / file_name).read_text().splitlines() if line.strip()]
    splits[split_name] = [line.split(",")[0].replace("_S1Hand.tif", "") for line in lines]

# Overlap check: each chip ID is expected to appear in exactly one split
all_ids = [chip for ids in splits.values() for chip in ids]
in_two_splits = sorted(chip for chip, n in Counter(all_ids).items() if n > 1)
print("Chips listed:", len(all_ids))
print("Unique chips:", len(set(all_ids)))
print("Chips in more than one split:", len(in_two_splits))

# File check: every listed chip is expected to have an S1, S2 and label file on Drive
missing = []
for chip in all_ids:
    for layer in LAYERS:
        if not (HAND_DIR / layer / f"{chip}_{layer}.tif").exists():
            missing.append(f"{chip}_{layer}.tif")
print("Missing files:", len(missing))

# Chips per flood event in each split (event = text before the first underscore)
events_per_split = {}
for split_name, ids in splits.items():
    counts = Counter(chip.split("_")[0] for chip in ids)
    events_per_split[split_name] = dict(sorted(counts.items()))
    print(f"\n{split_name} ({len(ids)} chips): {events_per_split[split_name]}")

# Stage 0 data report, saved as JSON on Drive
report = {
    "created": datetime.now().isoformat(timespec="seconds"),
    "source": "Sen1Floods11 v1.1, gs://sen1floods11/v1.1 (github.com/cloudtostreet/Sen1Floods11)",
    "files_per_layer": {layer: len(list((HAND_DIR / layer).glob("*.tif"))) for layer in LAYERS},
    "chips_per_split": {name: len(ids) for name, ids in splits.items()},
    "unique_chips": len(set(all_ids)),
    "chips_in_more_than_one_split": in_two_splits,
    "missing_files": missing,
    "events_per_split": events_per_split,
}
REPORT_PATH.write_text(json.dumps(report, indent=2))
print("\nReport saved:", REPORT_PATH)

Chips listed: 446
Unique chips: 446
Chips in more than one split: 0
Missing files: 0

train (252 chips): {'Ghana': 31, 'India': 40, 'Mekong': 18, 'Nigeria': 10, 'Pakistan': 16, 'Paraguay': 39, 'Somalia': 15, 'Spain': 18, 'Sri-Lanka': 24, 'USA': 41}

valid (89 chips): {'Ghana': 11, 'India': 14, 'Mekong': 6, 'Nigeria': 4, 'Pakistan': 6, 'Paraguay': 14, 'Somalia': 5, 'Spain': 6, 'Sri-Lanka': 9, 'USA': 14}

test (90 chips): {'Ghana': 11, 'India': 14, 'Mekong': 6, 'Nigeria': 4, 'Pakistan': 6, 'Paraguay': 14, 'Somalia': 6, 'Spain': 6, 'Sri-Lanka': 9, 'USA': 14}

bolivia (15 chips): {'Bolivia': 15}

Report saved: /content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/results/stage0_data_report.json
